# Check GPU

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# Install dependencies

In [2]:
!pip install -q tiktoken

# Imports

In [3]:
import math
import time
import os

import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


# GPT-2 configuration

In [4]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257

    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

# Causal Self Attention

In [5]:
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.n_embd = config.n_embd

        # Query, Key, Value projection
        self.c_attn = nn.Linear(
            config.n_embd,
            3 * config.n_embd
        )

        # Output projection
        self.c_proj = nn.Linear(
            config.n_embd,
            config.n_embd
        )

    def forward(self, x):

        B, T, C = x.size()

        # Calculate Q, K, V
        qkv = self.c_attn(x)

        q, k, v = qkv.split(
            self.n_embd,
            dim=2
        )

        # Split into attention heads

        q = q.view(
            B, T,
            self.n_head,
            C // self.n_head
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_head,
            C // self.n_head
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_head,
            C // self.n_head
        ).transpose(1, 2)

        # Causal self-attention
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        # Reassemble heads

        y = y.transpose(1, 2).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        # Output projection

        y = self.c_proj(y)

        return y

# MLP

In [6]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.c_fc = nn.Linear(
            config.n_embd,
            4 * config.n_embd
        )

        self.gelu = nn.GELU(
            approximate="tanh"
        )

        self.c_proj = nn.Linear(
            4 * config.n_embd,
            config.n_embd
        )

    def forward(self, x):

        x = self.c_fc(x)

        x = self.gelu(x)

        x = self.c_proj(x)

        return x

# Transformer Block

In [7]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(
            config.n_embd
        )

        self.attn = CausalSelfAttention(
            config
        )

        self.ln_2 = nn.LayerNorm(
            config.n_embd
        )

        self.mlp = MLP(
            config
        )

    def forward(self, x):

        # Attention + residual connection
        x = x + self.attn(
            self.ln_1(x)
        )

        # MLP + residual connection
        x = x + self.mlp(
            self.ln_2(x)
        )

        return x

# GPT model

In [8]:
class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config

        # Embeddings
        self.transformer = nn.ModuleDict({

            "wte": nn.Embedding(
                config.vocab_size,
                config.n_embd
            ),

            "wpe": nn.Embedding(
                config.block_size,
                config.n_embd
            ),

            "h": nn.ModuleList([
                Block(config)
                for _ in range(config.n_layer)
            ]),

            "ln_f": nn.LayerNorm(
                config.n_embd
            )
        })

        # Language model head
        self.lm_head = nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False
        )

        # Weight tying
        self.transformer["wte"].weight = \
            self.lm_head.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        idx,
        targets=None
    ):

        B, T = idx.size()

        assert T <= self.config.block_size

        # Position IDs

        pos = torch.arange(
            0,
            T,
            dtype=torch.long,
            device=idx.device
        )

        # Token embeddings

        tok_emb = self.transformer["wte"](
            idx
        )

        # Position embeddings

        pos_emb = self.transformer["wpe"](
            pos
        )

        # Combine

        x = tok_emb + pos_emb

        # Transformer blocks

        for block in self.transformer["h"]:
            x = block(x)

        # Final LayerNorm

        x = self.transformer["ln_f"](x)

        # Vocabulary logits

        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

# Count parameters

In [9]:
config = GPTConfig()

model = GPT(config)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Number of parameters: "
    f"{num_params:,}"
)

print(
    f"Parameters in millions: "
    f"{num_params / 1e6:.2f}M"
)

Number of parameters: 124,439,808
Parameters in millions: 124.44M


# Move model to GPU

In [10]:
model = model.to(device)

print(
    "Model is on:",
    next(model.parameters()).device
)

Model is on: cuda:0


# Prepare the dataset

In [11]:
!wget -O my_dataset.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open(
    "my_dataset.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()

print("Characters:", len(text))
print(text[:500])

--2026-09-26 07:58:26--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘my_dataset.txt’

my_dataset.txt      100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-09-26 07:58:26 (184 MB/s) - ‘my_dataset.txt’ saved [1115394/1115394]

Characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
N

# GPT-2 tokenizer

In [12]:
enc = tiktoken.get_encoding("gpt2")

tokens = enc.encode(text)

print("Number of tokens:", len(tokens))

print(tokens[:20])

Number of tokens: 338025
[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248]


# Train/validation split

In [13]:
tokens = torch.tensor(
    tokens,
    dtype=torch.long
)

split = int(
    0.9 * len(tokens)
)

train_tokens = tokens[:split]
val_tokens = tokens[split:]

print(
    "Training tokens:",
    len(train_tokens)
)

print(
    "Validation tokens:",
    len(val_tokens)
)

Training tokens: 304222
Validation tokens: 33803


# Data loader

In [14]:
batch_size = 4
block_size = 1024

def get_batch(
    data,
    batch_size,
    block_size,
    device
):

    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return (
        x.to(device),
        y.to(device)
    )

# Optimizer

In [15]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=6e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

# Training configuration

In [20]:
max_steps = 100

eval_interval = 250

eval_steps = 20

grad_accum_steps = 8

max_grad_norm = 1.0

# Learning-rate scheduler

In [21]:
max_lr = 6e-4
min_lr = 6e-5

warmup_steps = 500

def get_lr(step):

    # Warmup

    if step < warmup_steps:

        return max_lr * (
            step + 1
        ) / warmup_steps

    # After training

    if step >= max_steps:

        return min_lr

    # Cosine decay

    decay_ratio = (
        step - warmup_steps
    ) / (
        max_steps - warmup_steps
    )

    coeff = 0.5 * (
        1.0 +
        math.cos(
            math.pi * decay_ratio
        )
    )

    return (
        min_lr +
        coeff *
        (max_lr - min_lr)
    )

# Validation function

In [22]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    losses = {}

    for split_name, data in [
        ("train", train_tokens),
        ("val", val_tokens)
    ]:

        loss_list = []

        for _ in range(eval_steps):

            x, y = get_batch(
                data,
                batch_size,
                block_size,
                device
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ) if device == "cuda" else torch.no_grad():

                logits, loss = model(
                    x,
                    y
                )

            loss_list.append(
                loss.item()
            )

        losses[split_name] = sum(
            loss_list
        ) / len(loss_list)

    model.train()

    return losses

# Training loop

In [23]:
model.train()

start_time = time.time()

for step in range(max_steps):

    optimizer.zero_grad(
        set_to_none=True
    )

    loss_accum = 0.0

    # Gradient accumulation

    for micro_step in range(
        grad_accum_steps
    ):

        x, y = get_batch(
            train_tokens,
            batch_size,
            block_size,
            device
        )

        if device == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                logits, loss = model(
                    x,
                    y
                )

        else:

            logits, loss = model(
                x,
                y
            )

        loss = loss / grad_accum_steps

        loss_accum += loss.item()

        loss.backward()

    # Gradient clipping

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_grad_norm
    )

    # Learning rate

    lr = get_lr(step)

    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # Update parameters

    optimizer.step()

    # Logging

    if step % 10 == 0:

        elapsed = time.time() - start_time

        print(
            f"step {step:5d} | "
            f"loss {loss_accum:.4f} | "
            f"lr {lr:.2e} | "
            f"time {elapsed:.1f}s"
        )

    # Validation

    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"step {step}: "
            f"train loss = {losses['train']:.4f}, "
            f"val loss = {losses['val']:.4f}"
        )

step     0 | loss 5.4651 | lr 1.20e-06 | time 12.3s
step 0: train loss = 5.4826, val loss = 5.7870
step    10 | loss 5.4668 | lr 1.32e-05 | time 171.4s
step    20 | loss 5.4446 | lr 2.52e-05 | time 305.7s
step    30 | loss 5.3471 | lr 3.72e-05 | time 439.5s
step    40 | loss 5.2821 | lr 4.92e-05 | time 573.7s
step    50 | loss 5.1592 | lr 6.12e-05 | time 708.3s
step    60 | loss 5.0036 | lr 7.32e-05 | time 842.9s
step    70 | loss 4.8734 | lr 8.52e-05 | time 977.1s
step    80 | loss 4.7514 | lr 9.72e-05 | time 1111.6s
step    90 | loss 4.7609 | lr 1.09e-04 | time 1246.1s


# Generate text

In [24]:
@torch.no_grad()
def generate(
    model,
    prompt,
    max_new_tokens=100,
    temperature=1.0,
    top_k=50
):

    model.eval()

    tokens = enc.encode(prompt)

    x = torch.tensor(
        tokens,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    for _ in range(max_new_tokens):

        # Don't exceed context window

        x_cond = x[:, -block_size:]

        if device == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                logits, _ = model(x_cond)

        else:

            logits, _ = model(x_cond)

        # Last token

        logits = logits[:, -1, :]

        # Temperature

        logits = logits / temperature

        # Top-k sampling

        if top_k is not None:

            values, indices = torch.topk(
                logits,
                min(top_k, logits.size(-1))
            )

            logits_filtered = torch.full_like(
                logits,
                float("-inf")
            )

            logits_filtered.scatter_(
                1,
                indices,
                values
            )

            logits = logits_filtered

        # Probabilities

        probs = F.softmax(
            logits,
            dim=-1
        )

        # Sample

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Append

        x = torch.cat(
            [x, next_token],
            dim=1
        )

    model.train()

    return enc.decode(
        x[0].tolist()
    )

# Generate something

In [25]:
prompt = """
Once upon a time
"""

generated = generate(
    model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50
)

print(generated)


Once upon a time
And it speak, I to the time to myself to this man of war is the doth to the crown's the one
I'll in the king,
In your life,
And;


As we are be I may be the this,
And, be the king,
And the king's a king,
And all the gods'd the fault of the rest of mine, for the heart,
And,
And so he will I shall;

GLOUCESTER:
What, a I had I have his life

And now the king, now,
And,
To your king but by the state of his good time,


And,
The war, I will I must I to thee to the this that?
My good doth the king;

And this we have heard him.
So not the dost:

For I will.
With the prince,
You are in


# Save the model

In [26]:
checkpoint = {
    "model": model.state_dict(),
    "config": config,
    "optimizer": optimizer.state_dict(),
    "step": max_steps
}

torch.save(
    checkpoint,
    "gpt2_colab.pt"
)

print("Model saved!")

Model saved!


# Save to Google Drive

In [27]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

# Google Drive path
save_path = "/content/drive/MyDrive/gpt2_colab.pt"

checkpoint = {
    "model": model.state_dict(),
    "config": config,
    "optimizer": optimizer.state_dict(),
    "step": max_steps
}

torch.save(checkpoint, save_path)

print(f"Model saved to: {save_path}")
print(f"File size: {os.path.getsize(save_path) / (1024**3):.2f} GB")

Mounted at /content/drive
Model saved to: /content/drive/MyDrive/gpt2_colab.pt
File size: 1.39 GB
